In [7]:
from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass, field
from typing import TypeAlias

import numpy as np
from numpy.typing import NDArray

In [8]:
# TYPES
# Canonical internal array representation
FloatArray: TypeAlias = NDArray[np.float64]

# Tensor metadata
TensorType: TypeAlias = tuple[int, int]
Index: TypeAlias = int | tuple[int, ...]

# Accepted numeric scalar types at the library boundary
NumericScalar: TypeAlias = int | float | np.integer | np.floating

# Accepted raw inputs before canonical conversion
VectorData: TypeAlias = Sequence[NumericScalar] | FloatArray
MatrixData: TypeAlias = Sequence[Sequence[NumericScalar]] | FloatArray
TensorData: TypeAlias = NumericScalar | VectorData | MatrixData

In [9]:
# VALIDATIONS
def validate_tensor_type(tensor_type: TensorType) -> None:
    """
    Validate that tensor_type is a pair of non-negative integers (r, s).
    """
    if not isinstance(tensor_type, tuple):
        raise TypeError(
            f"tensor_type must be a tuple[int, int], got {type(tensor_type).__name__}."
        )

    if len(tensor_type) != 2:
        raise ValueError(
            f"tensor_type must have length 2, got length {len(tensor_type)}."
        )

    r, s = tensor_type

    if not isinstance(r, int) or not isinstance(s, int):
        raise TypeError(
            f"tensor_type entries must be integers, got ({type(r).__name__}, {type(s).__name__})."
        )

    if r < 0 or s < 0:
        raise ValueError(
            f"tensor_type entries must be non-negative, got {tensor_type}."
        )


def validate_float_array(arr: object) -> FloatArray:
    """
    Validate that arr is a NumPy array with dtype float64.
    Return the validated array cast as FloatArray.
    """
    if not isinstance(arr, np.ndarray):
        raise TypeError(
            f"Expected a NumPy ndarray, got {type(arr).__name__}."
        )

    if arr.dtype != np.float64:
        raise TypeError(
            f"Expected dtype float64, got {arr.dtype}."
        )

    return arr


def validate_rank_matches_type(
    components: FloatArray,
    tensor_type: TensorType,
) -> None:
    """
    Validate that the array rank matches the tensor type (r, s).
    """
    expected_rank = sum(tensor_type)
    actual_rank = components.ndim

    if actual_rank != expected_rank:
        raise ValueError(
            f"Tensor type {tensor_type} requires rank {expected_rank}, "
            f"but components have rank {actual_rank} and shape {components.shape}."
        )


def validate_vector_shape(arr: FloatArray) -> None:
    """
    Validate that arr is 1-dimensional.
    """
    if arr.ndim != 1:
        raise ValueError(
            f"Expected a 1D array, got rank {arr.ndim} with shape {arr.shape}."
        )


def validate_matrix_shape(arr: FloatArray) -> None:
    """
    Validate that arr is 2-dimensional.
    """
    if arr.ndim != 2:
        raise ValueError(
            f"Expected a 2D array, got rank {arr.ndim} with shape {arr.shape}."
        )

In [10]:
# ARRAY VALIDATION AND CONVERSION
def is_numeric_scalar(x: object) -> bool:
    """
    Return True if x is a numeric scalar accepted by the library.
    """
    return (
        isinstance(x, (int, float, np.integer, np.floating))
        and not isinstance(x, bool)
    )


def to_float_array(data: TensorData) -> FloatArray:
    """
    Convert raw tensor data into the canonical internal representation:
    a NumPy ndarray with dtype float64.
    """
    if isinstance(data, (str, bytes)):
        raise TypeError("Strings and bytes are not valid tensor data.")

    if is_numeric_scalar(data):
        return np.asarray(data, dtype=np.float64)

    if isinstance(data, np.ndarray):
        if data.dtype.kind not in {"i", "u", "f"}:
            raise TypeError(
                f"NumPy array must contain numeric data, got dtype {data.dtype}."
            )
        return np.asarray(data, dtype=np.float64)

    if not isinstance(data, Sequence):
        raise TypeError(
            "Tensor data must be a numeric scalar, a numeric sequence, "
            "or a numeric nested sequence."
        )

    try:
        arr = np.asarray(data, dtype=np.float64)
    except (TypeError, ValueError) as exc:
        raise TypeError(
            "Tensor data could not be converted to a float64 NumPy array."
        ) from exc

    if arr.dtype != np.float64:
        raise TypeError(
            f"Expected conversion to dtype float64, got {arr.dtype}."
        )

    return arr

In [11]:
# TENSOR CLASS
@dataclass(frozen=True)
class Tensor:
    """
    General tensor represented by a float64 NumPy array together with its tensor type.
    """

    components: FloatArray
    tensor_type: TensorType

    def __post_init__(self) -> None:
        validate_tensor_type(self.tensor_type)
        validate_float_array(self.components)
        validate_rank_matches_type(self.components, self.tensor_type)

    @property
    def rank(self) -> int:
        """
        Return the total tensor rank, equal to the number of array axes.
        """
        return self.components.ndim

    @property
    def shape(self) -> tuple[int, ...]:
        """
        Return the shape of the underlying NumPy array.
        """
        return self.components.shape

    def __getitem__(self, index: Index):
        """
        Access tensor components by index.
        """
        return self.components[index]

    def __repr__(self) -> str:
        return (
            f"Tensor(tensor_type={self.tensor_type}, "
            f"rank={self.rank}, shape={self.shape})\n"
            f"{self.components}"
        )

    @classmethod
    def from_data(cls, data: TensorData, tensor_type: TensorType) -> "Tensor":
        """
        Build a tensor from raw input data by converting it to the canonical float64 array form.
        """
        components = to_float_array(data)
        return cls(components=components, tensor_type=tensor_type)

In [12]:
# VECTOR CLASS
@dataclass(frozen=True)
class Vector(Tensor):
    """
    Contravariant tensor of type (1, 0).
    """

    tensor_type: TensorType = field(init=False, default=(1, 0))

    def __post_init__(self) -> None:
        super().__post_init__()
        validate_vector_shape(self.components)

    @classmethod
    def from_data(cls, data: VectorData) -> "Vector":
        """
        Build a vector from raw input data.
        """
        components = to_float_array(data)
        return cls(components=components)

In [13]:
# COVECTOR CLASS

@dataclass(frozen=True)
class Covector(Tensor):
    """
    Covariant tensor of type (0, 1).
    """

    tensor_type: TensorType = field(init=False, default=(0, 1))

    def __post_init__(self) -> None:
        super().__post_init__()
        validate_vector_shape(self.components)

    @classmethod
    def from_data(cls, data: VectorData) -> "Covector":
        """
        Build a covector from raw input data.
        """
        components = to_float_array(data)
        return cls(components=components)

In [14]:
# TESTS AND USAGE EXAMPLES

# --- Tensor from raw data ---
t = Tensor.from_data([[1, 2], [3, 4]], tensor_type=(1, 1))
print("Tensor t:")
print(t)
print("t.tensor_type =", t.tensor_type)
print("t.rank =", t.rank)
print("t.shape =", t.shape)
print()

# --- Vector ---
v = Vector.from_data([1, 2, 3])
print("Vector v:")
print(v)
print("v.tensor_type =", v.tensor_type)
print("v.rank =", v.rank)
print("v.shape =", v.shape)
print("v[1] =", v[1])
print("isinstance(v, Vector) =", isinstance(v, Vector))
print("isinstance(v, Tensor) =", isinstance(v, Tensor))
print()

# --- Covector ---
alpha = Covector.from_data([10, 20, 30])
print("Covector alpha:")
print(alpha)
print("alpha.tensor_type =", alpha.tensor_type)
print("alpha.rank =", alpha.rank)
print("alpha.shape =", alpha.shape)
print("alpha[2] =", alpha[2])
print("isinstance(alpha, Covector) =", isinstance(alpha, Covector))
print("isinstance(alpha, Tensor) =", isinstance(alpha, Tensor))
print()

# --- Same shape, different tensor meaning ---
print("Same shape, different tensor type:")
print("v.shape =", v.shape, "| v.tensor_type =", v.tensor_type)
print("alpha.shape =", alpha.shape, "| alpha.tensor_type =", alpha.tensor_type)
print()

# --- Direct construction from canonical arrays ---
v2 = Vector(np.array([4.0, 5.0, 6.0], dtype=np.float64))
alpha2 = Covector(np.array([7.0, 8.0, 9.0], dtype=np.float64))

print("Direct construction from canonical arrays:")
print(v2)
print(alpha2)
print()

# --- Error cases ---
print("Expected errors:")
try:
    Vector.from_data([[1, 2], [3, 4]])
except Exception as exc:
    print("Vector.from_data([[1, 2], [3, 4]]) ->", type(exc).__name__, exc)

try:
    Covector.from_data("abc")
except Exception as exc:
    print('Covector.from_data("abc") ->', type(exc).__name__, exc)

try:
    Tensor.from_data([1, 2, 3], tensor_type=(1, 1))
except Exception as exc:
    print("Tensor.from_data([1, 2, 3], tensor_type=(1, 1)) ->", type(exc).__name__, exc)

Tensor t:
Tensor(tensor_type=(1, 1), rank=2, shape=(2, 2))
[[1. 2.]
 [3. 4.]]
t.tensor_type = (1, 1)
t.rank = 2
t.shape = (2, 2)

Vector v:
Vector(components=array([1., 2., 3.]), tensor_type=(1, 0))
v.tensor_type = (1, 0)
v.rank = 1
v.shape = (3,)
v[1] = 2.0
isinstance(v, Vector) = True
isinstance(v, Tensor) = True

Covector alpha:
Covector(components=array([10., 20., 30.]), tensor_type=(0, 1))
alpha.tensor_type = (0, 1)
alpha.rank = 1
alpha.shape = (3,)
alpha[2] = 30.0
isinstance(alpha, Covector) = True
isinstance(alpha, Tensor) = True

Same shape, different tensor type:
v.shape = (3,) | v.tensor_type = (1, 0)
alpha.shape = (3,) | alpha.tensor_type = (0, 1)

Direct construction from canonical arrays:
Vector(components=array([4., 5., 6.]), tensor_type=(1, 0))
Covector(components=array([7., 8., 9.]), tensor_type=(0, 1))

Expected errors:
Vector.from_data([[1, 2], [3, 4]]) -> ValueError Tensor type (1, 0) requires rank 1, but components have rank 2 and shape (2, 2).
Covector.from_data("a